# RAG 평가셋 v2

기존 `rag_eval.ipynb`와 분리된 평가 파이프라인입니다. RAGAS 6개 지표는 오프라인 품질 평가로 사용하고, ID 기반 검색 지표와 인용 검증으로 검색 실패·생성 실패·유효 보류를 분리합니다. `EvalConfig.score_thresholds`는 팀 기준 확정 전의 기본값입니다.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.evaluation.rag_eval_v2 import *
config = EvalConfig()
config

In [ ]:
# 최종 통합 평가셋만 사용합니다.
GOLDEN_PATH = ROOT/'data/evaluation/golden_set_final.jsonl'
cases = [row for row in read_jsonl(GOLDEN_PATH) if row.get('status', 'READY') == 'READY']
chunks = read_jsonl(ROOT/'data/processed/chunks.jsonl')
chunk_by_id = {c.get('chunk_id'): c for c in chunks if c.get('chunk_id')}
validate_cases(cases, chunk_by_id)
print(f'golden_set={GOLDEN_PATH}, cases={len(cases)}, chunks={len(chunks)}')

## 구조화 답변 생성기

아래 어댑터는 실제 Answer Agent로 교체할 자리입니다. 반드시 `status`, `answer`, `citations`를 반환해야 하며, `source_url`은 원문 값이 없으면 `null`로 둡니다. 인용 검증은 문자열 존재 검사를 먼저 수행하고, 실제 운영에서는 `support_checker`를 별도 LLM 판정기로 연결합니다.

In [ ]:
def answer_generator(question, retrieved_chunks):
    # TODO: 프로젝트 Answer Agent 호출로 교체
    if not retrieved_chunks:
        return {'status':'INSUFFICIENT_EVIDENCE', 'answer':'확인할 수 없습니다.', 'citations':[]}
    c = retrieved_chunks[0]
    quote = context_text(c)
    return {'status':'ANSWERED', 'answer':quote, 'citations':[{'claim':quote, 'quote':quote, 'chunk_id':c['chunk_id'], 'source_url':c.get('source_url')}] }

# 별도 LLM entailment 판정기로 교체 가능. 현재는 보수적으로 문자열 기반 결과를 사용합니다.
def support_checker(claim, quote):
    return bool(claim and quote and claim in quote)


## RAGAS 6개 지표

RAGAS 입력의 `reference`는 반드시 golden set의 `reference` 필드입니다. `reference_contexts`와 `expected_source_chunk_ids`는 ID 기반 지표 및 원문 확인에 사용하며, RAGAS 의미 지표가 직접 사용한다고 가정하지 않습니다. 아래 셀은 기존 RAGAS 설정을 연결하기 위한 어댑터입니다.

In [ ]:
# 실제 RAGAS 실행 결과를 case_id -> metric dict로 넣으세요.
# 예: ragas_rows = {'CASE1': {'faithfulness': .9, ...}}
ragas_rows = {}
try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, ContextPrecision, ContextRecall, ContextRelevancy
    print('RAGAS imports OK; connect your existing evaluator here.')
except Exception as e:
    print('RAGAS import skipped:', e)


In [ ]:
results = evaluate_cases(cases, chunks, answer_generator, ragas_rows=ragas_rows, config=config, support_checker=support_checker)
summary = summarize(results, config)
out_path, summary_path = save_results(results, summary, ROOT)
print(out_path)
print(summary_path)
print(json.dumps({k: summary[k] for k in ['overall_pass','pass_rate','failure_counts','id_metrics']}, ensure_ascii=False, indent=2))

In [ ]:
# 실패 문항 원문 확인: 검색 원문과 정답 원문이 결과 JSONL에 함께 저장됩니다.
failed = [r for r in results if not r['passed']]
for r in failed[:10]:
    print(r['case_id'], r['failure_cause'], r['retrieved_chunk_ids'], r['expected_chunk_ids'])
    print('retrieved:', r['retrieved_source_texts'])
    print('expected:', r['expected_source_texts'])